# PhiState: bounded training correction and native sampled chat

Run all to compare the saved optimizer policy with protected recurrent parameters, evaluate generated answers, and export only a qualifying candidate. Maximum **1,000 additional updates total**. Existing Drive runs are preserved. Training uses PyTorch/CUDA; inference uses native RFTMW C++ CPU. This fixes execution controls; it does not guarantee that the checkpoint becomes a capable assistant.

The data mixture stays fixed for this optimizer comparison. Qualification covers 32 development and 32 separate final functional smoke tasks, not broad assistant certification.

T4 and L4 runtimes are supported by the FP32 path. Before training, the notebook checks actual-checkpoint CPU/GPU forward and backward agreement and reports GPU memory. No BF16 support is required.

In [ ]:
from pathlib import Path
import importlib.util, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
BRANCH = 'feat/phistate-coherence-execution-v1'
REPO = 'https://github.com/Anharmoniclabs/quantoniumoslinux.git'
CACHE = Path('/content/phistate_coherence_git')
if not CACHE.exists():
    subprocess.run(['git', 'clone', '--bare', '--filter=blob:none', '--single-branch', '--branch', BRANCH, REPO, str(CACHE)], check=True)
subprocess.run(['git', '-C', str(CACHE), 'fetch', 'origin', BRANCH], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(CACHE), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
CHECKOUT = Path('/content') / ('phistate_coherence_' + COMMIT[:12])
if not CHECKOUT.exists():
    subprocess.run(['git', '-C', str(CACHE), 'worktree', 'add', '--detach', str(CHECKOUT), COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip() == COMMIT
missing = [name for name in ['sentencepiece', 'numpy'] if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime in Colab.'
assert hasattr(torch.optim, 'Adafactor'), 'This checkpoint requires PyTorch with torch.optim.Adafactor.'
LAB = CHECKOUT / 'runtime/experiments/phi_runtime_lab'
ROOT = Path('/content/drive/MyDrive/Quantonium/PhiState-L4/whitespace_repair_v1')
print('CODE COMMIT:', COMMIT)
print('TRAINING:', torch.cuda.get_device_name(0), '| native inference: RFTMW C++ CPU')


In [ ]:
# One fixed comparison: at most 500 extra updates per arm, then qualification.
command = [sys.executable, '-u', str(LAB / 'coherence_recovery.py'),
           '--root', str(ROOT), '--device', 'cuda', '--steps', '500', '--clear-stop']
process = subprocess.Popen(command)
try:
    process.wait()
except KeyboardInterrupt:
    (ROOT / 'coherence_execution_v1/STOP').touch()
    print('Save-and-stop requested. Waiting for the current complete update to save.')
    process.wait()
assert process.returncode == 0, 'Worker failed; its last atomic checkpoint is retained.'


In [ ]:
import json
report_path = ROOT / 'coherence_execution_v1/report.json'
if report_path.exists():
    RESULT = json.loads(report_path.read_text())
    print('RESULT:', RESULT['status'])
    for arm in RESULT.get('arms', []):
        progress = arm['progress']
        print(arm['policy'], '| completed updates:', progress['phase_step'],
              '| best development score:', progress['best_development_score'])
        for source, coverage in progress['coverage'].items():
            print(' ', source, '| draws:', coverage['draws'], '| unique:', coverage['unique_examples'],
                  '| assistant targets:', coverage['answer_tokens'])
    if RESULT['status'] == 'qualified':
        sys.path.insert(0, str(LAB))
        from chat_recovery_cpp import chat
        chat(Path(RESULT['native_run']), decode_config=dict(temperature=.7, top_p=.9, top_k=50,
             repetition_penalty=1.1, seed=20261031))
    else:
        print('No candidate promoted. Results and checkpoints are saved; this budget will not extend automatically.')
else:
    print('The experiment paused before a comparison report. Rerun the fixed run cell to resume.')
